# 🤝 Challenge 19: beanllm × Message Broker 시너지 가이드

**난이도:** ⭐⭐⭐⭐⭐  
**실행 요구사항:** 브로커 랩(`docker compose up -d`) — beanllm은 **선택 사항**

---

## 이 노트북의 목적

> **"beanllm을 만든 사람이 메시지 브로커를 공부하면 무엇을 얻는가?"
> 그리고 "메시지 브로커를 알면 beanllm의 내부가 어떻게 보이는가?"**

- beanllm의 각 기능이 내부적으로 어떤 **브로커 패턴**을 사용하는지 매핑
- 브로커 랩에서 직접 실습한 패턴이 **프로덕션 AI 시스템에서 어떻게 쓰이는지** 연결
- 실제 브로커 API 호출로 AI 파이프라인의 각 구간을 **직접 체험**

### ⚡ 실행 방법
- **섹션 1~4**: 브로커 랩만 실행 중이면 됩니다 (beanllm 불필요)
- **섹션 5**: beanllm 설치 시 실제 LLM 연동 가능 (`pip install beanllm[ollama]`)
- 코드 블록 앞에 `# [브로커만]` 또는 `# [beanllm 필요]` 표시

## 아키텍처 전체 그림


| beanllm (AI 두뇌 — 고수준) | message-broker-comparison-lab (AI 신경계 — 저수준) |
| :--- | :--- |
| `RAGChain.from_documents()` | → Redis Vector Set (`VADD`/`VSIM`) |
| `EmbeddingClient.embed()` | → Redis Cache (`SET`/`GET` with TTL) |
| `Agent.run()` 병렬 처리 | → RabbitMQ Task Queue |
| LLM 스트리밍 응답 | → Redis Stream / Kafka |
| `rate_limit` 데코레이터 | → Redis Rate Limiter (`ZADD`) |
| 완료 알림 콜백 | → RabbitMQ Direct Queue |
| 대량 문서 임베딩 배치 | → Kafka Batch Produce |
| 결과 이벤트 소싱 | → Kafka Consumer Group |



**핵심 통찰**: beanllm은 이 모든 저수준 패턴을 추상화한다.  
브로커 랩을 배우면 → beanllm이 왜 그런 설계를 했는지 이해할 수 있다.

In [ ]:
# [브로커만] 환경 설정
import asyncio
import json
import time
import httpx
from datetime import datetime

BASE_URL = "http://localhost:8000"

async def call(method: str, path: str, **kwargs):
    async with httpx.AsyncClient(base_url=BASE_URL, timeout=30.0) as client:
        resp = await getattr(client, method)(path, **kwargs)
        return resp.json()

# 헬스체크
health = await call("get", "/health")
print("✅ 브로커 연결 상태:")
for k, v in health.items():
    icon = "🟢" if v in ("running", "connected") else "🔴"
    print(f"  {icon} {k}: {v}")

---

## 📦 섹션 1: RAGChain 내부 구조 — Redis Vector Set 직접 실습

### beanllm에서는...

```python
# beanllm 코드 (참고용 — 실행 불필요)
from beanllm import RAGChain

rag = RAGChain.from_documents("docs/")
result = await rag.query("메시지 브로커 비교")
# 내부적으로:
# 1. EmbeddingClient로 쿼리 벡터화
# 2. VectorStore에서 코사인 유사도 검색  ← 바로 이게 Redis VSIM!
# 3. 관련 문서를 LLM 컨텍스트로 전달
```

### 브로커 랩에서 직접 구현하면...

Challenge 18에서 배운 Redis VSIM이 RAGChain의 핵심입니다.

**아래 코드는 RAGChain 내부의 Vector Search를 직접 보여줍니다:**

In [ ]:
# [브로커만] RAGChain 내부 패턴 — Redis Vector 직접 시연
import redis.asyncio as aioredis
import numpy as np
import hashlib

r = aioredis.from_url("redis://localhost:6379", decode_responses=False)
r_text = aioredis.from_url("redis://localhost:6379", decode_responses=True)

DIM = 64  # 간소화된 차원 (실제 beanllm: text-embedding-3-small = 1536차원)

# ─── Step 1: 문서 임베딩 (Mock) ───
# beanllm에서: EmbeddingClient.embed(text)
# 여기서: 해시 기반 Mock (브로커 패턴 학습이 목적)
def mock_embed(text: str) -> np.ndarray:
    seed = int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)
    vec = np.random.RandomState(seed).randn(DIM).astype(np.float32)
    return vec / np.linalg.norm(vec)

# ─── Step 2: 문서 색인 (RAGChain.from_documents 내부) ───
docs = [
    {"id": "doc_kafka",    "text": "Kafka는 분산 로그 기반 메시지 브로커로 대용량 스트리밍에 적합합니다"},
    {"id": "doc_redis",    "text": "Redis는 인메모리 DB로 초저지연 캐시와 Pub/Sub에 사용됩니다"},
    {"id": "doc_rabbit",   "text": "RabbitMQ는 AMQP 프로토콜 기반으로 신뢰성 있는 메시지 전달을 보장합니다"},
    {"id": "doc_rag",      "text": "RAG는 LLM이 외부 문서를 참조하여 답변 정확도를 높이는 기법입니다"},
    {"id": "doc_vector",   "text": "벡터 검색은 텍스트 의미를 수치 벡터로 변환해 유사도를 계산합니다"},
]

VKEY = "rag:doc_vectors"
await r.delete(VKEY)

for doc in docs:
    vec = mock_embed(doc["text"])
    try:
        await r.execute_command("VADD", VKEY, "REDUCE", DIM, "FP32", vec.tobytes(), doc["id"])
        await r_text.set(f"rag:doc:{doc['id']}", doc["text"])
    except Exception as e:
        if "unknown command" in str(e).lower():
            print("⚠️  Redis 8.4+ 필요 (VADD 미지원). Redis 버전을 확인하세요.")
            break

print("📚 문서 색인 완료 (RAGChain.from_documents 내부 동작)")
print(f"  저장된 벡터: {len(docs)}개 → Redis VCARD: ", end="")
try:
    cnt = await r.execute_command("VCARD", VKEY)
    print(cnt)
except:
    print("(VSIM 미지원 Redis)")

# ─── Step 3: 쿼리 검색 (RAGChain.query 내부) ───
query = "메시지 브로커를 선택하는 기준은 무엇인가요?"
q_vec = mock_embed(query)

print(f"\n🔍 쿼리: '{query}'")
print("\n  Top-3 유사 문서 (VSIM 결과):")

try:
    results = await r.execute_command("VSIM", VKEY, "FP32", q_vec.tobytes(), "COUNT", 3, "WITHSCORES")
    for i in range(0, len(results), 2):
        doc_id = results[i].decode()
        score = float(results[i+1])
        text = await r_text.get(f"rag:doc:{doc_id}")
        print(f"  [{score:.4f}] {doc_id}: {text[:50]}...")
except Exception as e:
    print(f"  ℹ️  VSIM 미지원: {e}")
    print("  → Challenge 18 노트북을 Redis 8.4+ 환경에서 실행해보세요")

print("\n💡 이것이 beanllm RAGChain.query()의 내부 동작입니다!")
await r.aclose()
await r_text.aclose()

---

## 🔴 섹션 2: beanllm Rate Limiter — Redis 패턴 직접 구현

### beanllm에서는...

```python
# beanllm 코드 (참고용)
from beanllm import Client

# rate_limit 데코레이터: 동일 사용자 분당 20회 제한
client = Client(model="gpt-4o", rate_limit={"per_minute": 20})
response = await client.chat(messages=[...])  
# 초과 시 → RateLimitError 발생
```

### 브로커 랩 Redis Rate Limiter와 완전히 동일한 원리

```text
[beanllm 내부]              [브로커 랩 /redis/ratelimit/check]

ZADD user:rate <ts> <req_id>   ←→   ZADD rate:<user> <ts> <uuid>
ZREMRANGEBYSCORE ... 60초전         ZREMRANGEBYSCORE ... 60초전  
ZCARD user:rate > 20 → 차단         ZCARD > limit → 429 반환
```

In [ ]:
# [브로커만] Rate Limiter 직접 체험 — beanllm 내부 동작과 동일한 패턴

# 브로커 랩의 Rate Limiter API 호출
user_id = "beanllm-demo-user"
limit = 5  # 데모용: 분당 5회 제한

print(f"⚡ Rate Limiter 테스트 (limit: {limit}회/분)")
print("="*50)

for i in range(7):  # 제한 초과 시도
    resp = await call("get", f"/redis/ratelimit/check",
                      params={"user_id": user_id, "limit": limit})
    
    allowed = resp.get("allowed", False)
    count = resp.get("current_count", "?")
    icon = "✅" if allowed else "🚫"
    
    print(f"  요청 #{i+1:2d}: {icon} {'허용' if allowed else '차단'} "
          f"(현재 카운트: {count}/{limit})")
    
    if not allowed:
        print(f"\n  💡 beanllm에서는 이 시점에 RateLimitError가 발생합니다")
        print(f"     → 자동 재시도 or 사용자에게 '잠시 후 다시 시도' 안내")

print("\n📌 학습 포인트:")
print("  beanllm의 rate_limit 파라미터는 내부적으로")
print("  Sliding Window Counter (ZADD + ZREMRANGEBYSCORE)를 사용합니다")
print("  → 이것이 바로 Challenge 09에서 배운 Redis Rate Limiter 패턴!")

---

## 🟠 섹션 3: beanllm Agent 병렬 실행 — RabbitMQ Task Queue 패턴

### beanllm에서는...

```python
# beanllm 코드 (참고용)
from beanllm import Agent, Tool

@Tool.from_function
async def analyze_order(order_id: str) -> str:
    """주문을 분석합니다"""
    return f"주문 {order_id} 분석 완료"

agent = Agent(model="gpt-4o-mini", tools=[analyze_order])

# 병렬 Agent 실행 (내부적으로 작업 큐 패턴 사용)
results = await asyncio.gather(*[
    agent.run(f"주문 {i}번을 분석해줘")
    for i in range(10)
])
```

### 이 패턴의 브로커 버전

```text
beanllm Agent 병렬 실행           ←→   RabbitMQ Work Queue 패턴

  Producer (주문 요청)                   LPUSH / RPUSH
  Worker Pool (Agent 인스턴스들)    ←→   BRPOP (여러 Consumer)
  결과 수집                              RabbitMQ ACK
  실패 재처리                            Dead Letter Queue (DLQ)
```

**아래는 RabbitMQ로 beanllm Agent 병렬 처리 패턴을 시뮬레이션합니다:**

In [ ]:
# [브로커만] RabbitMQ Task Queue = beanllm Agent 작업 분배 시뮬레이션

import asyncio

TASK_QUEUE = "agent-task-queue"
RESULT_QUEUE = "agent-result-queue"

# ─── Producer: 10개 AI 작업을 큐에 등록 ───
# beanllm에서: asyncio.gather()로 여러 agent.run() 동시 호출
# 브로커에서: RabbitMQ Direct Queue에 작업 등록

tasks = [
    {"task_id": f"task_{i:03d}", "type": "analyze",
     "payload": f"주문 {i}번: {['신발', '티셔츠', '모자', '가방', '지갑'][i%5]} 구매",
     "priority": "high" if i < 3 else "normal"}
    for i in range(10)
]

print("📤 작업 큐에 AI Task 등록 중...")
start = time.time()

for task in tasks:
    await call("post", "/rabbitmq/direct/publish",
               params={"queue_name": TASK_QUEUE},
               json={"content": json.dumps(task, ensure_ascii=False),
                     "metadata": {"type": "agent_task", "id": task["task_id"]}})

queue_info = await call("get", f"/rabbitmq/queue/info/{TASK_QUEUE}")
enqueued = queue_info.get("message_count", len(tasks))

print(f"  ✅ {len(tasks)}개 작업 등록 완료 → 큐 대기중: {enqueued}개")
print(f"  ⏱  등록 시간: {(time.time()-start)*1000:.1f}ms")

# ─── 학습 포인트 ───
print("\n📌 beanllm Agent 병렬 처리 vs 브로커 Task Queue")
print("┌──────────────────────────────────────────────────┐")
print("│ beanllm: asyncio.gather() → 메모리에서 병렬      │")
print("│ 브로커:  RabbitMQ Queue → 프로세스 간 분산        │")
print("│                                                  │")
print("│ beanllm 장점: 단순, 빠름                          │")
print("│ 브로커 장점: 내구성, 재시도, 여러 서버에 분산        │")
print("└──────────────────────────────────────────────────┘")
print("\n💡 프로덕션: beanllm Agent가 RabbitMQ를 통해 작업을 받으면")
print("   서버를 재시작해도 큐에 남아있는 작업이 유실되지 않음!")

---

## 🔵 섹션 4: LLM 스트리밍 응답 — Kafka Stream 패턴

### beanllm에서는...

```python
# beanllm 스트리밍 응답 (참고용)
from beanllm import Client

client = Client(model="gpt-4o")

# 토큰이 생성될 때마다 스트리밍
async for chunk in client.stream_chat(
    messages=[{"role": "user", "content": "메시지 브로커를 설명해줘"}]
):
    print(chunk, end="", flush=True)  # 토큰 단위로 화면에 출력
```

### 이것이 Kafka + Redis Stream의 결합 패턴


<div align="center">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 200" width="100%" height="auto">
  <defs>
    <filter id="shadow" x="-5%" y="-5%" width="110%" height="110%">
      <feDropShadow dx="1" dy="2" stdDeviation="2" flood-color="#000" flood-opacity="0.1" />
    </filter>
  </defs>
  <rect width="100%" height="100%" fill="#F8FAFC" rx="12" />

  <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">LLM 토큰 스트리밍 파이프라인 (Kafka / Redis Stream)</text>

  <!-- LLM API -->
  <g transform="translate(50, 60)" filter="url(#shadow)">
    <rect x="0" y="0" width="140" height="90" rx="8" fill="#F1F5F9" stroke="#64748B" stroke-width="1.5" />
    <text x="70" y="35" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#334155" text-anchor="middle">LLM API</text>
    <text x="70" y="60" font-family="-apple-system, sans-serif" font-size="10" fill="#64748B" text-anchor="middle">토큰 단위 실시간 생성</text>
  </g>

  <!-- Arrow 1 -->
  <path d="M 190 105 L 240 105" stroke="#64748B" stroke-width="2" />
  <text x="215" y="95" font-family="monospace" font-size="8" fill="#475569" text-anchor="middle">XADD</text>

  <!-- Broker Stream -->
  <g transform="translate(250, 60)" filter="url(#shadow)">
    <rect x="0" y="0" width="220" height="90" rx="8" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" />
    <text x="110" y="25" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#166534" text-anchor="middle">Stream (Redis / Kafka)</text>
    
    <!-- Stream queue representation -->
    <g transform="translate(20, 45)" font-family="monospace" font-size="9" fill="#14532D">
      <rect x="0" y="0" width="40" height="25" rx="3" fill="#BBF7D0" stroke="#15803D" /><text x="20" y="16" text-anchor="middle">"메시지"</text>
      <rect x="45" y="0" width="40" height="25" rx="3" fill="#BBF7D0" stroke="#15803D" /><text x="65" y="16" text-anchor="middle">"브로커"</text>
      <rect x="90" y="0" width="40" height="25" rx="3" fill="#BBF7D0" stroke="#15803D" /><text x="110" y="16" text-anchor="middle">"란"</text>
      <rect x="135" y="0" width="40" height="25" rx="3" fill="#BBF7D0" stroke="#15803D" /><text x="155" y="16" text-anchor="middle">"분산"</text>
    </g>
  </g>

  <!-- Arrow 2 -->
  <path d="M 470 105 L 520 105" stroke="#15803D" stroke-width="2" />
  <text x="495" y="95" font-family="monospace" font-size="8" fill="#166534" text-anchor="middle">XREAD</text>

  <!-- Consumer / UI -->
  <g transform="translate(530, 60)" filter="url(#shadow)">
    <rect x="0" y="0" width="220" height="90" rx="8" fill="#E0F2FE" stroke="#0284C7" stroke-width="1.5" />
    <text x="110" y="25" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#0369A1" text-anchor="middle">Consumer (FastAPI) &amp; UI</text>
    <rect x="15" y="45" width="190" height="30" rx="4" fill="#FFFFFF" stroke="#CBD5E1" stroke-width="1" />
    <text x="25" y="64" font-family="monospace" font-size="11" fill="#0F172A">메시지 브로커란 분산...</text>
    <circle cx="190" cy="60" r="3" fill="#22C55E" />
  </g>
</svg>
</div>



In [ ]:
# [브로커만] LLM 스트리밍 시뮬레이션 — Redis Stream으로 토큰 스트리밍

import redis.asyncio as aioredis

r = aioredis.from_url("redis://localhost:6379", decode_responses=True)

STREAM_KEY = "llm:token:stream"
SESSION_ID = f"sess_{int(time.time())}"

# LLM이 생성하는 토큰 시뮬레이션
simulated_tokens = [
    "메시지", " 브로커는", " 분산", " 시스템에서",
    " 컴포넌트", " 간", " 비동기", " 통신을",
    " 담당하는", " 핵심", " 인프라입니다.",
    " Kafka는", " 대용량", " 스트리밍에",
    " Redis는", " 초저지연에",
    " RabbitMQ는", " 신뢰성에", " 특화되어", " 있습니다.",
    "[END]"
]

print(f"🤖 LLM 스트리밍 시뮬레이션 (Session: {SESSION_ID})")
print("="*55)
print("\n[Producer — LLM 토큰 생성 및 Redis Stream에 추가]")

# ─── Producer: 토큰을 Redis Stream에 순서대로 추가 ───
stream_key = f"{STREAM_KEY}:{SESSION_ID}"
await r.delete(stream_key)

for i, token in enumerate(simulated_tokens):
    msg_id = await r.xadd(stream_key, {
        "token": token,
        "index": str(i),
        "done": "true" if token == "[END]" else "false"
    })
    await asyncio.sleep(0.05)  # LLM 생성 속도 시뮬레이션

print(f"  {len(simulated_tokens)}개 토큰 → Redis Stream 완료")

# ─── Consumer: Stream에서 실시간으로 읽어서 출력 ───
print("\n[Consumer — 사용자 화면에 실시간 렌더링]")
print("\n생성된 답변: ", end="")

last_id = "0"
while True:
    messages = await r.xread({stream_key: last_id}, count=5, block=100)
    if not messages:
        break
    for stream, msgs in messages:
        for msg_id, fields in msgs:
            token = fields.get("token", "")
            if token == "[END]":
                print("\n")
                break
            print(token, end="", flush=True)
            last_id = msg_id
        else:
            continue
        break
    else:
        continue
    break

# Stream 정보
info = await r.xinfo_stream(stream_key)
print(f"\n📊 Stream 통계:")
print(f"  총 메시지: {info['length']}개 (토큰)")
print(f"  첫 ID: {info['first-entry'][0]}")
print(f"  마지막 ID: {info['last-entry'][0]}")

print("\n💡 beanllm stream_chat()은 내부적으로 이 패턴으로 구현됩니다:")
print("   LLM API → 토큰 수신 → Redis/Kafka Stream → Client 전달")

await r.aclose()

---

## 🚀 섹션 5: beanllm 실제 연동 (선택 — 리소스 여유 시)

> ⚠️ **이 섹션은 beanllm 설치 + LLM API 키가 필요합니다**  
> 리소스가 부족하면 이 섹션은 건너뛰세요. 위 섹션들로 충분히 학습됩니다.

### 설치
```bash
# Ollama 로컬 모델 (무료, API 키 불필요)
pip install beanllm[ollama]
ollama pull nomic-embed-text   # 임베딩용
ollama pull llama3.2           # 추론용

# 또는 API 사용
pip install beanllm[openai]
export OPENAI_API_KEY=sk-...
```

In [ ]:
# [beanllm 필요] — 설치된 경우에만 실행

try:
    from beanllm import EmbeddingClient, Agent, Tool
    BEANLLM_AVAILABLE = True
    print("✅ beanllm 설치됨 — 실제 임베딩 및 Agent 사용 가능")
except ImportError:
    BEANLLM_AVAILABLE = False
    print("⚠️  beanllm 미설치 — 이 섹션은 건너뜁니다")
    print("   위 섹션 1~4의 내용으로 패턴 이해는 완료되었습니다! ✨")

In [ ]:
# [beanllm 필요] 실제 Embedding → Redis Vector Set

if BEANLLM_AVAILABLE:
    import redis.asyncio as aioredis
    
    # Ollama 로컬 임베딩 (nomic-embed-text: 768차원)
    # 또는 OpenAI text-embedding-3-small (1536차원)
    embed_client = EmbeddingClient(model="ollama/nomic-embed-text")
    
    r = aioredis.from_url("redis://localhost:6379", decode_responses=False)
    r_text = aioredis.from_url("redis://localhost:6379", decode_responses=True)
    
    VKEY_REAL = "rag:real_vectors"
    await r.delete(VKEY_REAL)
    
    docs_real = [
        {"id": "kafka",   "text": "Kafka는 분산 로그 기반 메시지 브로커입니다"},
        {"id": "redis",   "text": "Redis는 인메모리 데이터 구조 서버입니다"},
        {"id": "rabbitmq","text": "RabbitMQ는 AMQP 메시지 브로커입니다"},
    ]
    
    print("📐 실제 임베딩 생성 중 (beanllm EmbeddingClient)...")
    for doc in docs_real:
        vec = await embed_client.embed(doc["text"])
        vec_np = np.array(vec, dtype=np.float32)
        dim = len(vec_np)
        
        await r.execute_command("VADD", VKEY_REAL, "REDUCE", dim, "FP32", vec_np.tobytes(), doc["id"])
        await r_text.set(f"rag:real:{doc['id']}", doc["text"])
        print(f"  ✓ {doc['id']}: {dim}차원 벡터 저장")
    
    # 실제 유사도 검색
    query = "메시지를 빠르게 처리하는 시스템은?"
    q_vec = await embed_client.embed(query)
    q_np = np.array(q_vec, dtype=np.float32)
    
    results = await r.execute_command("VSIM", VKEY_REAL, "FP32", q_np.tobytes(), "COUNT", 3, "WITHSCORES")
    
    print(f"\n🔍 쿼리: '{query}'")
    for i in range(0, len(results), 2):
        doc_id = results[i].decode()
        score = float(results[i+1])
        text = await r_text.get(f"rag:real:{doc_id}")
        print(f"  [{score:.4f}] {doc_id}: {text}")
    
    print("\n✨ Mock 임베딩 vs 실제 임베딩 비교:")
    print("  Mock: 해시 기반 → 의미 무관한 랜덤 벡터")
    print("  실제: 의미 기반 → 유사한 텍스트는 가까운 벡터")
    
    await r.aclose()
    await r_text.aclose()
else:
    print("⏭  건너뜀 (beanllm 미설치)")

In [ ]:
# [beanllm 필요] Agent + Tool로 브로커 API 제어

if BEANLLM_AVAILABLE:
    import httpx
    
    # ─── 브로커 API를 beanllm Tool로 래핑 ───
    @Tool.from_function
    async def get_broker_health() -> str:
        """메시지 브로커 전체 연결 상태를 확인합니다"""
        async with httpx.AsyncClient(base_url=BASE_URL, timeout=5) as c:
            resp = await c.get("/health")
            data = resp.json()
            return f"브로커 상태: Redis={data.get('redis')}, RabbitMQ={data.get('rabbitmq')}, Kafka={data.get('kafka')}"

    @Tool.from_function
    async def run_broker_benchmark(broker: str, message_count: int = 500) -> str:
        """특정 브로커의 성능 벤치마크를 실행합니다. broker는 redis/kafka/rabbitmq 중 하나입니다"""
        async with httpx.AsyncClient(base_url=BASE_URL, timeout=60) as c:
            resp = await c.post(f"/benchmark/{broker}", json={"message_count": message_count})
            data = resp.json()
            return (f"{broker} 벤치마크 결과: "
                    f"{data.get('total_messages')}개, "
                    f"처리량={data.get('throughput_msg_per_sec', 0):.0f} msg/s, "
                    f"평균지연={data.get('avg_latency_ms', 0):.2f}ms")

    @Tool.from_function
    async def get_kafka_topics() -> str:
        """Kafka 토픽 목록을 조회합니다"""
        async with httpx.AsyncClient(base_url=BASE_URL, timeout=5) as c:
            resp = await c.get("/kafka/topics")
            data = resp.json()
            return f"Kafka 토픽: {data.get('topic_count')}개 — {data.get('topics', [])[:5]}"

    # ─── beanllm Agent가 브로커를 자연어로 제어 ───
    agent = Agent(
        model="ollama/llama3.2",  # 또는 "gpt-4o-mini"
        tools=[get_broker_health, run_broker_benchmark, get_kafka_topics]
    )

    print("🤖 beanllm Agent가 브로커를 자연어로 제어합니다!")
    print("="*55)
    
    result = await agent.run(
        "Redis와 Kafka의 현재 상태를 확인하고, 각각 300개 메시지로 "
        "벤치마크를 실행한 뒤 어느 것이 더 빠른지 비교해줘"
    )
    
    print(f"\nAgent 결론:\n{result}")
else:
    print("⏭  건너뜀 (beanllm 미설치)")
    print("\n📌 위 코드의 핵심 패턴 요약:")
    print("  @Tool.from_function → 함수를 AI가 호출 가능한 도구로 변환")
    print("  Agent.run('자연어') → LLM이 필요한 Tool을 자동 선택 및 실행")
    print("  결과 → 자연어로 종합 분석")
    print("")
    print("  브로커 랩의 FastAPI 엔드포인트가 beanllm Tool의 구현체!")

---

## 📊 전체 정리: beanllm × 브로커 패턴 매핑표

| beanllm 기능 | 내부 브로커 패턴 | 브로커 랩 위치 |
|-------------|----------------|---------------|
| `RAGChain.query()` | Redis VSIM 코사인 유사도 | Challenge 18, 섹션 1 |
| `EmbeddingClient.embed()` + 저장 | Redis VADD | Challenge 18 |
| `rate_limit` 데코레이터 | Redis ZADD Sliding Window | `/redis/ratelimit/check` |
| `Agent` 병렬 Task | RabbitMQ Work Queue | Challenge 11, 14 |
| `stream_chat()` 토큰 스트리밍 | Redis Stream XADD/XREAD | Challenge 13, 17 |
| 완료 콜백/알림 | RabbitMQ Direct Queue | Challenge 16 |
| 대량 문서 색인 배치 | Kafka Batch Produce | Challenge 14 |
| 추론 요청 큐잉 | Kafka Topic + Consumer Group | Challenge 18 MISS 처리 |
| GraphRAG 관계 탐색 | — (Neo4j 직접, 브로커 X) | — |

## 🎯 핵심 메시지

> **beanllm을 더 잘 쓰려면 브로커 패턴을 알아야 하고,  
> 브로커 패턴을 배우면 beanllm의 설계 의도가 보인다.**

이 두 프로젝트는 같은 문제를 서로 다른 레벨에서 해결합니다:
- **beanllm**: "AI가 뭘 해야 하나" (비즈니스 로직, 고수준)
- **브로커 랩**: "AI 결과가 어떻게 전달되나" (인프라, 저수준)

## 다음 단계

- [→ 20. 대용량 처리 튜닝](./20_high_throughput_tuning.ipynb) — 실제 1M 메시지 처리 실험
- [← 18. AI 시맨틱 캐시](./18_challenge_ai_semantic_cache.ipynb) — Redis Vector Set 심화